In [1]:
import torch
from config import get_config
from model import BigramLanguageModel
from load_data import load_data

config = get_config()
train_data, val_data, vocab_size, decode = load_data()

In [2]:
device = config["device"]
model = BigramLanguageModel(vocab_size)
m = model.to(device)
optimizer = torch.optim.AdamW(m.parameters(), lr=config["learning_rate"])

In [3]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - config["block_size"], (config["batch_size"],))
    x = torch.stack([data[i:i+config["block_size"]] for i in ix])
    y = torch.stack([data[i+1:i+1+config["block_size"]] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_losses():
    out = {}
    model.eval() #Setting model to eval phase
    for split in ["train", "val"]:
        losses = torch.zeros(config["eval_iters"])
        for k in range(config["eval_iters"]):
            X, y = get_batch(split)
            logit, loss = model(X, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() #Resetting model to training phase
    return out

In [4]:
for iter in range(config["max_iters"]):

    if iter%config["eval_interval"]==0:
        losses = estimate_losses()
        print(f"step {iter}: train loss: {losses['train']:.4f}, val losses: {losses['val']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss: 4.3126, val losses: 4.3162
step 100: train loss: 2.6356, val losses: 2.6442
step 200: train loss: 2.5034, val losses: 2.4995
step 300: train loss: 2.4129, val losses: 2.4307
step 400: train loss: 2.3698, val losses: 2.3811
step 500: train loss: 2.3267, val losses: 2.3329
step 600: train loss: 2.2742, val losses: 2.2864
step 700: train loss: 2.2368, val losses: 2.2606
step 800: train loss: 2.2027, val losses: 2.2270
step 900: train loss: 2.1758, val losses: 2.1904
step 1000: train loss: 2.1418, val losses: 2.1727
step 1100: train loss: 2.1044, val losses: 2.1442
step 1200: train loss: 2.0979, val losses: 2.1384
step 1300: train loss: 2.0479, val losses: 2.0969
step 1400: train loss: 2.0364, val losses: 2.0900
step 1500: train loss: 2.0127, val losses: 2.0650
step 1600: train loss: 1.9934, val losses: 2.0400
step 1700: train loss: 1.9644, val losses: 2.0349
step 1800: train loss: 1.9606, val losses: 2.0295
step 1900: train loss: 1.9446, val losses: 2.0036
step 2000: t

In [5]:
print("Generating from the model: \n\n")

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

Generating from the model: 



To my of exult of the you se weet aburthed?

RISERSSS:
Her more.
Ruren yourncesy, Sproved diderer-say.

FRIAR LAR BINCELIUS:
I hon, our loves, gently the muke this smancliss?
O the my tall dimper: I'll hose unsCous:
Where the mast hust pronding: adve that you,
Yethis I if it, spoke shall long,
Sirk'd not drect themer a afty wantent abole.

BUTUS:
Hathe thing I sent know eyes: the steand ir.

QURIONEN:
If my nurtlew mhe! Lord my coldiedingt and by you takem:
Well, the patk yet-belings a cael of to beare;
But onif Capirles self it not crepety.

LO&ZETE:
I but theil:
We, lark you, he doth ere Within that think I
will a proceous the vowery thee ladgetty, that up; I from elf the fight thou daw do ad made resesit to Shaigold? hy serpincer; hose we think he your held frienthedd; I,
ASWARANNA:
I my knunds, unsing loves, this lurdider'd in the vis,
Ary?
They doth the beisome for a in the reace,
Hich dangms force, I, my'e puch, sourdh,
It slaw the courgiant and ser